# 02 - Preprocessing and Split Validation

**Goal:** verify that `src/preprocess.py` (crop black borders, resize to 512x512, CLAHE on the L channel) worked on all 3,504 cleaned images, and that the train/validation split from `src/make_split.py` is correct.

**Input:** `data/clean.csv` (from notebook 01). **Output checked:** `data/processed_512/` and `data/splits/`.

## 1. Checking the output folder
Confirms that `processed_512` contains exactly one file per kept image, and that every image is 512x512.

In [14]:
import cv2, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA = "../data"
PROC = f"{DATA}/processed_512"
clean = pd.read_csv(f"{DATA}/clean.csv")

files = os.listdir(PROC)
print("Images in clean.csv:", len(clean))
print("Files in processed_512:", len(files))
print("Every clean image has a file:", set(clean["id_code"] + ".png") == set(files))

shapes = {cv2.imread(f"{PROC}/{f}").shape for f in files[:300]}
print("Shapes found (first 300 files):", shapes)

Images in clean.csv: 3504
Files in processed_512: 3504
Every clean image has a file: True
Shapes found (first 300 files): {(512, 512, 3)}


**Interpretation:** the folder contains 3,504 files, one per image in `clean.csv`, so no image was skipped. All checked images have shape (512, 512, 3).

## 2. Before/after comparison
Five random images: raw (top) versus preprocessed (bottom).

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for j, (_, r) in enumerate(clean.sample(5, random_state=7).iterrows()):
    raw = cv2.cvtColor(cv2.imread(f"{DATA}/train_images/{r.id_code}.png"), cv2.COLOR_BGR2RGB)
    proc = cv2.cvtColor(cv2.imread(f"{PROC}/{r.id_code}.png"), cv2.COLOR_BGR2RGB)
    axes[0, j].imshow(raw);  axes[0, j].set_title(f"Raw {raw.shape[:2]}")
    axes[1, j].imshow(proc); axes[1, j].set_title(f"Processed {proc.shape[:2]} | grade {r.diagnosis}")
    axes[0, j].axis("off"); axes[1, j].axis("off")
plt.tight_layout(); plt.show()

**Interpretation:** all five processed images are 512x512, whatever the raw size (from 1050x1050 up to 1958x2588). The black borders are removed and the retina fills the frame. Colors look natural (orange/red retinas), so the BGR/RGB conversions are correct, and the bluish-cyan tint of the last image is already present in the raw photo, so it comes from the camera and not from the code. After CLAHE, vessels and small lesions are more visible, for example the exudate cluster in the third image. 

**Visible limitation:** raw images with a 4:3 shape (the 2nd to 4th pairs) are compressed horizontally when forced to a square, so the retina looks slightly oval. This is an accepted simplification, applied identically to every image.

## 3. Brightness statistics (automatic outlier detection)
Checking 3,504 images by eye is impossible, so we compute the mean pixel value of each processed image (0 = black, 255 = white) and look at the distribution and the extremes.

In [16]:
clean["mean_brightness"] = [cv2.imread(f"{PROC}/{c}.png").mean() for c in clean["id_code"]]
print(clean["mean_brightness"].describe().round(1))
print()
print(clean.nsmallest(5, "mean_brightness")[["id_code", "diagnosis", "mean_brightness"]])

count    3504.0
mean       74.7
std        12.2
min        33.3
25%        67.5
50%        75.5
75%        82.0
max       117.2
Name: mean_brightness, dtype: float64

           id_code  diagnosis  mean_brightness
1645  77baa08a1345          2        33.305500
3002  d91635f380b4          0        34.682875
2939  d4583e9525dc          0        35.630146
2529  b6304c545f95          2        37.331022
1756  7f6690fa390a          0        37.536170


**Interpretation:** mean = 74.7, median = 75.5, std = 12.2, min = 33.3, max = 117.2 (middle half of the images between 67.5 and 82.0). Mean and median are almost equal, so the distribution is balanced, and no value is close to 0, so there is no black or failed image.

## 4. Visual check of the 5 darkest and 5 brightest processed images

In [ ]:
darkest = clean.nsmallest(5, "mean_brightness")
brightest = clean.nlargest(5, "mean_brightness")

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
for row, subset, label in [(0, darkest, "Dark"), (1, brightest, "Bright")]:
    for j, (_, r) in enumerate(subset.iterrows()):
        img = cv2.cvtColor(cv2.imread(f"{PROC}/{r.id_code}.png"), cv2.COLOR_BGR2RGB)
        axes[row, j].imshow(img)
        axes[row, j].set_title(f"{label} | grade {r.diagnosis} | {r.mean_brightness:.0f}")
        axes[row, j].axis("off")
plt.tight_layout(); plt.show()

**Interpretation:** all ten images are valid retinas with correct cropping. The five darkest (brightness 33 to 38) are dim, but vessels, optic disc and macula remain visible, and the two grade 2 images still show small yellowish exudates. The five brightest (115 to 117) are pale or overexposed, but vessels and lesions stay sharp: the second image (grade 2) shows many small bright spots and dark red patches, typical signs of diabetic retinopathy. The last bright image has a bluish tint caused by the camera. The darkest images were already among the images flagged "dark" or "blurry" in notebook 01 and were kept after visual review. No image is removed at this stage.

## 5. Train/validation split

The split was created once by `src/make_split.py`: stratified by grade, 80/20, seed 42, from the deduplicated list. We verify its size, class proportions and the absence of overlap.

In [18]:
train = pd.read_csv(f"{DATA}/splits/train.csv")
val = pd.read_csv(f"{DATA}/splits/val.csv")
report = pd.read_csv(f"{DATA}/image_report.csv").set_index("id_code")

print("Train:", len(train), "| Val:", len(val))
print("Overlap by id:  ", len(set(train.id_code) & set(val.id_code)))
print("Overlap by hash:", len(set(report.loc[train.id_code, "hash"]) & set(report.loc[val.id_code, "hash"])))

tab = pd.DataFrame({"train": train.diagnosis.value_counts().sort_index(),
                    "val": val.diagnosis.value_counts().sort_index()})
tab["val_%"] = (100 * tab["val"] / (tab["train"] + tab["val"])).round(1)
tab.index = ["No DR", "Mild", "Moderate", "Severe", "Prolif."]
print(tab)

Train: 2803 | Val: 701
Overlap by id:   0
Overlap by hash: 0
          train  val  val_%
No DR      1437  359   20.0
Mild        270   68   20.1
Moderate    737  185   20.1
Severe      142   35   19.8
Prolif.     217   54   19.9


**Interpretation**
- **2,803 training and 701 validation images**, from the 3,504 cleaned images (80/20, stratified by grade, seed 42).
- **No overlap, by id or by image hash**: no validation image has an identical copy in the training set. In the first version of the project, 41 duplicate groups were split across both sets, which inflated the validation scores.
- Validation share per grade is 20% for every grade, so the stratification worked.

| Grade | Train | Val |
|---|---|---|
| 0 No DR | 1437 | 359 |
| 1 Mild | 270 | 68 |
| 2 Moderate | 737 | 185 |
| 3 Severe | 142 | 35 |
| 4 Proliferative | 217 | 54 |

- **Limitation:** the validation set contains only 35 severe and 54 proliferative images, so one image changes the grade-3 recall by about 2.9 points. Per-grade metrics for the rare grades will be very noisy.

## Summary

| Aspect | Result |
|---|---|
| Preprocessed images | 3,504, one per cleaned image, all 512x512 |
| Steps | crop black borders, resize, CLAHE on the L channel |
| Visual check | all inspected images are valid retinas, correctly cropped, natural colors; mild horizontal compression of 4:3 images |
| Split | 2,803 train / 701 validation, stratified, seed 42, no overlap by id or hash |

**Next:** training (`03_training.ipynb`) with the v2 configuration: pretrained EfficientNet-B0, weighted sampler (1/sqrt(count)), label smoothing 0.1, weight decay 1e-2, 15 epochs.